In [ ]:
import Shadow
import logging
import numpy as np
import matplotlib.pyplot as plt
import statistics
from random import random
from ophyd import Signal, Component, Device
from ophyd.signal import InternalSignal
from optlnls.plot import plot_beam
from optlnls.importing import read_shadow_beam
from bluesky import RunEngine
from bluesky.plans import scan
from bluesky.callbacks.best_effort import BestEffortCallback
from bluesky.callbacks.tiled_writer import TiledWriter
from tiled.client import simple
from blop.plans import optimize
from blop.protocols import EvaluationFunction, Optimizer, OptimizationProblem

RE = RunEngine()

tiled_client = simple()
tiled_writer = TiledWriter(tiled_client)
RE.subscribe(tiled_writer)

bec = BestEffortCallback()
bec.disable_plots()
RE.subscribe(bec)

logging.getLogger("httpx").setLevel(logging.WARNING)

## Ophyd Devices Integrated with Shadow3

In [ ]:
class BeamStats(Device):
    center_x = Component(InternalSignal, kind="hinted")
    center_y = Component(InternalSignal, kind="hinted")
    width = Component(InternalSignal, kind="hinted")
    height = Component(InternalSignal, kind="hinted")
    image = Component(InternalSignal, kind="hinted")
    original_image_shape = (0, 0)

    def read(self, **kwargs):
        beam_info = self.beam.histo2(col_h = 3, col_v = 1, nbins_h = 100, nbins_v = 100, nolost = 0, ref = 23)
        self.center_x.put((beam_info['xrange'][1] + beam_info['xrange'][0])/2, internal=True)
        self.center_y.put((beam_info['yrange'][1] + beam_info['yrange'][0])/2, internal=True)
        self.width.put(beam_info['fwhm_h'], internal=True)
        self.height.put(beam_info['fwhm_v'], internal=True)
        beam2D = read_shadow_beam(
                beam=self.beam,
                x_column_index=3,
                y_column_index=1
        )
        self.original_image_shape = beam2D.shape
        self.image.put(beam2D.ravel(), internal=True)
        return super().read(**kwargs)

In [ ]:
class ShadowEnergy(Signal):

    def set(self, value: float, **kwargs):
        sts = super().set(value, **kwargs)
        self.parent.oe.PH1 = value - self.parent.delta_energy / 2
        self.parent.oe.PH2 = value + self.parent.delta_energy / 2
        return sts

class ShadowAttribute(Signal):

    def set(self, value: float, **kwargs):
        sts = super().set(value, **kwargs)
        oe = self.parent.set_standard_parameters(self.parent.energy.get())
        self.parent.oe = oe
        return sts


class ShadowSource(BeamStats):

    energy = Component(ShadowEnergy, kind="hinted")
    sigmax = Component(ShadowAttribute, kind="hinted")

    def __init__(
        self, name: str, energy: float, delta_energy: float, n_rays: int = 1_000_000
    ):
        super().__init__(name=name)
        self.delta_energy = delta_energy
        self.n_rays = n_rays
        self.oe = self.set_standard_parameters(energy)
        self.energy.set(energy).wait()
        self.sigmax.set(0.033).wait()

    def set_standard_parameters(self, energy):
        oe = Shadow.Source()
        oe.SIGDIX = 0.01
        oe.SIGDIZ = 0.00113
        oe.SIGMAZ = 0.027
        oe.SIGMAX = self.sigmax.get()
        oe.VDIV1 = 0.001
        oe.VDIV2 = 0.001
        oe.FDISTR = 3
        oe.F_COLOR = 3
        oe.F_PHOT = 0
        oe.HDIV1 = 0.004
        oe.HDIV2 = 0.004
        oe.IDO_VX = 0
        oe.IDO_VZ = 0
        oe.IDO_X_S = 0
        oe.IDO_Y_S = 0
        oe.IDO_Z_S = 0
        oe.ISTAR1 = 5676561
        oe.NPOINT = self.n_rays
        oe.PH1 = energy - self.delta_energy / 2
        oe.PH2 = energy + self.delta_energy / 2
        return oe

    def read(self, **kwargs):
        self.beam = Shadow.Beam()
        self.beam.genSource(self.oe)
        # beam2D = read_shadow_beam(
        #         beam=self.beam,
        #         x_column_index=3,
        #         y_column_index=1
        # )
        # plot_beam(
        #     beam2D=beam2D,
        #     show_plot=True,
        #     fitType=3,
        #     textA=2,textB=5,
        #     x_range=1,x_range_min=-0.4,x_range_max=0.3,
        #     y_range=1,y_range_min=-0.4,y_range_max=0.3,
        #     zero_pad_x=2
        # )
        return super().read(**kwargs)


In [ ]:
class ShadowM1(BeamStats):
    def __init__(self, past_element: list, **kwargs):
        super().__init__(**kwargs)
        self.past_element = past_element
        self.idx = 0
        self.generate_oe()

    def generate_oe(self):
        oe = Shadow.OE()
        oe.ALPHA = 90.0
        oe.CIL_ANG = 90.0
        oe.DUMMY = 0.1
        oe.FCYL = 1
        oe.FHIT_C = 1
        oe.FMIRR = 1
        oe.FWRITE = 1
        oe.F_EXT = 1
        oe.RLEN1 = 180.0
        oe.RLEN2 = 180.0
        oe.RMIRR = 2108.8753452152337
        oe.RWIDX1 = 20.0
        oe.RWIDX2 = 20.0
        oe.T_IMAGE = 35.0
        oe.T_INCIDENCE = 80.825
        oe.T_REFLECTION = 80.825
        oe.T_SOURCE = 0.0
        self.oe = oe

    def read(self, **kwargs):
        self.generate_oe()
        self.past_element.read()
        self.beam = self.past_element.beam
        self.beam.traceOE(self.oe, self.idx)
        # beam2D = read_shadow_beam(
        #         beam=self.beam,
        #         x_column_index=3,
        #         y_column_index=1
        # )
        # plot_beam(
        #     beam2D=beam2D,
        #     show_plot=True,
        #     fitType=3,
        #     textA=2,textB=5,
        #     x_range=1,x_range_min=-0.4,x_range_max=0.3,
        #     y_range=1,y_range_min=-0.4,y_range_max=0.3,
        #     zero_pad_x=2
        # )
        return super().read(**kwargs)

In [ ]:
class ShadowPitch(Signal):
    def describe(self, **kwargs):
        desc = super().describe()
        desc[self.name]["precision"] = 6
        return desc
    
    def set(self, value: float, **kwargs):
        sts = super().set(value, **kwargs)
        self.parent.generate_oe(value)
        return sts

class ShadowM2(BeamStats):
    pitch = Component(ShadowPitch, kind="hinted")

    def __init__(self, past_element: list, pitch: float, **kwargs):
        super().__init__(**kwargs)
        self.idx = 1
        self.past_element = past_element
        self.pitch.set(pitch)

    def generate_oe(self, pitch: float):
        oe = Shadow.OE()
        oe.ALPHA = 180.0
        oe.DUMMY = 0.1
        oe.FCYL = 1
        oe.FHIT_C = 1
        oe.FMIRR = 1
        oe.FWRITE = 1
        oe.F_DEFAULT = 0
        oe.F_MOVE = 1
        oe.RLEN1 = 300.0
        oe.RLEN2 = 300.0
        oe.RWIDX1 = 20.0
        oe.RWIDX2 = 20.0
        oe.SIMAG = 14095.0
        oe.SSOUR = 14095.0
        oe.THETA = 82.75
        oe.T_IMAGE = 19.05
        oe.T_INCIDENCE = 82.75
        oe.T_REFLECTION = 82.75
        oe.T_SOURCE = 0.0
        oe.X_ROT = np.degrees(pitch)
        self.oe = oe

    
    def read(self, **kwargs):
        self.generate_oe(self.pitch.get())
        self.past_element.read()
        self.beam = self.past_element.beam
        self.beam.traceOE(self.oe, self.idx)
        # beam2D = read_shadow_beam(
        #         beam=self.beam,
        #         x_column_index=3,
        #         y_column_index=1
        # )
        # plot_beam(
        #     beam2D=beam2D,
        #     show_plot=True,
        #     fitType=3,
        #     textA=2,textB=5,
        #     x_range=1,x_range_min=-0.4,x_range_max=0.3,
        #     y_range=1,y_range_min=-0.4,y_range_max=0.3,
        #     zero_pad_x=2
        # )
        return super().read(**kwargs)

## Beamline instantiation

In [ ]:
source = ShadowSource(
    name="src",
    energy=20000,
    delta_energy=1000,
    n_rays=100_000
)

m1 = ShadowM1(
    name="m1",
    past_element=source
)

m2 = ShadowM2(
        name="m2",
        pitch=0.003,
        past_element=m1
)
# m2.read()

## Optimization Code

In [ ]:
class DetectorEvaluation(EvaluationFunction):
    def __init__(self, tiled_client):
        self.tiled_client = tiled_client
    
    def __call__(self, uid: str, suggestions: list[dict]) -> list[dict]:
        outcomes = []
        run = self.tiled_client[uid]
        pitch = run["primary/internal/m2_pitch"].read()
        center_x = run["primary/internal/m2_center_x"].read()
        suggestion_ids = run.metadata["start"]["blop_suggestion_ids"]
        for idx, sid in enumerate(suggestion_ids):
            outcome = {
                "_id": sid,
                "center_x": center_x[idx],
                'pitch': pitch[idx]
            }
            outcomes.append(outcome)
        return outcomes

In [ ]:
def triangle_step(value, center, range):
    if value > center and value < (center + range):
        return -(value - center)/range + 1
    elif value <= center and value > (center - range):
        return (value - center)/range + 1
    return 0

class FuzzyEvaluation(EvaluationFunction):
    def __init__(self, tiled_client):
        self.tiled_client = tiled_client
        self.center = 0
        self.inference_method = self.larsen
        self.relation_method = max # (Additive method) sum
        self.defuzzify_method = self.max_defuzzify
        self.input_functions = {
            "center_x": {
                "bad_low": lambda val: triangle_step(val, self.center-0.4, 0.4),
                "good": lambda val: triangle_step(val, self.center, 0.4),
                "bad_high": lambda val: triangle_step(val, self.center+0.4, 0.4)
            }
        }
        self.output_functions = {
            "quality": {
                "good": lambda val: triangle_step(val, self.center, 1),
                "bad": lambda val: triangle_step(val, self.center+1, 1)
            }
        }

    #--------------- PLOTS --------------------
    
    def plot_input_functions(self, input_parameters):
        fuzzy_space = np.linspace(-1, 1, num=100)
        for input_key, _ in input_parameters.items():
            for _, function in self.input_functions[input_key].items():
                plt.scatter(fuzzy_space, [function(val) for val in fuzzy_space])
        plt.show()

    def plot_output_functions(self, fuzzy_inference):
        fuzzy_space = np.linspace(-1, 2, num=100)
        for inference_key, inference_values in fuzzy_inference.items():
            for key, function in self.output_functions[inference_key].items():
                plt.scatter(fuzzy_space, [function(val) for val in fuzzy_space])
                plt.scatter(fuzzy_space, [self.inference_method(inference_values[key], function(val)) for val in fuzzy_space])
        plt.show()

    def plot_inference(self, output_function4plot, centroid):
        centroid_space = np.linspace(0, 1, num=50)
        fuzzy_space = np.linspace(-1, 2, num=100)
        for output_function in output_function4plot:
            plt.scatter(fuzzy_space, output_function)
        plt.scatter([centroid for _ in centroid_space], centroid_space)
        plt.show()
    
    #------------------------------------------

    #--------------- INFERENCE METHODS --------------------

    def larsen(self, val, threshold_val):
        return val * threshold_val
    
    def mamdani(self, val, threshold_val):
        return min(val, threshold_val)

    #------------------------------------------------------

    #--------------- DEFUZZYFICATION METHODS --------------------

    def centroid(self, x_coord, y_coord):
        area_list = []
        mean_list = []
        for x in range(len(x_coord)):
            mean_x = (x_coord[x] + x_coord[x-1])/2
            diff_x = abs(y_coord[x] - y_coord[x-1])
            diff_y = abs(y_coord[x] - y_coord[x-1])/2
            triang_area = (diff_y*diff_x)/2
            square_area = diff_x*(min([y_coord[x], y_coord[x-1]]))
            total_area = square_area + triang_area
            mean_list.append(mean_x*total_area)
            area_list.append(total_area)
        return sum(mean_list)/sum(area_list)

    def max_defuzzify(self, x_coord, y_coord):
        # Can't have a max plateau
        return x_coord[list(y_coord).index(max(y_coord))]

    #------------------------------------------------------------


    def fuzzify(self, input_parameters):
        fuzzy_input = {}
        for input_key, input_val in input_parameters.items():
            fuzzy_input[input_key] = {}
            for attribute, function in self.input_functions[input_key].items():
                fuzzy_input[input_key][attribute] = function(input_val)
        self.plot_input_functions(input_parameters)
        return fuzzy_input

    def inference(self, fuzzy_values):
        bad_quality = max(
            [fuzzy_values["center_x"]["bad_low"], 
            fuzzy_values["center_x"]["bad_high"]])
        good_quality = max(
            [fuzzy_values["center_x"]["good"]])
        inference_value = {
            "quality": {
                "good": good_quality,
                "bad": bad_quality
            }
        }
        return inference_value

    def defuzzify(self, fuzzy_inference):
        fuzzy_output = {}
        fuzzy_space = np.linspace(-1, 2, num=100)
        self.plot_output_functions(fuzzy_inference)
        for inference_key, inference_values in fuzzy_inference.items():
            fuzzy_output[inference_key] = {}
            output_function = []
            output_function4plot = []
            for val in fuzzy_space:
                point_values = []
                for key, function in self.output_functions[inference_key].items():
                    point_values.append(self.inference_method(inference_values[key], function(val)))
                output_function.append(self.relation_method(point_values))
                output_function4plot.append(output_function)
            fuzzy_output[inference_key] = self.defuzzify_method(fuzzy_space, output_function)
            self.plot_inference(output_function4plot, fuzzy_output[inference_key])
        return fuzzy_output

    def calculate_cost(self, cost_parameters):
        fuzzy_values = self.fuzzify(cost_parameters)
        fuzzy_inference = self.inference(fuzzy_values)
        fuzzy_output = self.defuzzify(fuzzy_inference)
        return fuzzy_output["quality"]
        

    def __call__(self, uid: str, suggestions: list[dict]) -> list[dict]:
        outcomes = []
        run = self.tiled_client[uid]
        pitch = run["primary/internal/m2_pitch"].read()
        center_x = run["primary/internal/m2_center_x"].read()
        suggestion_ids = run.metadata["start"]["blop_suggestion_ids"]
        for idx, sid in enumerate(suggestion_ids):
            cost_parameters = {
                "center_x": center_x[idx]
            }
            cost = self.calculate_cost(cost_parameters)
            outcome = {
                "_id": sid,
                "center_x": center_x[idx],
                "pitch": pitch[idx],
                "cost": cost
            }
            outcomes.append(outcome)
        return outcomes

In [ ]:
class SearchOptimizer(Optimizer):

    verified_points = []
    map_old2new_outcome = {}
    outcome = set()
    dof = set()
    idx = 0

    def __init__(self):
        self.pitch_dof = [-0.01, 0.01]

    def convert_to_dof(self, value: float):
        nrange = abs(self.pitch_dof[1] - self.pitch_dof[0])
        return (value * nrange) + self.pitch_dof[0]

    def already_tested(self, value: float):
        for point in self.verified_points:
            if np.isclose(value, point, atol=self.step/10):
                return True
        return False

    def first_points(self, num_points: int | None = None):
        self.verified_points = []
        self.map_old2new_outcome = {}
        self.step = 0.001
        self.idx = 0
        self.num_points = num_points
        sug_points = []
        for _ in range(num_points):
            self.idx += 1
            pitch = self.convert_to_dof(random())
            self.verified_points.append(pitch)
            sug_points.append({
                "_id": self.idx,
                "m2_pitch": pitch
            })

        return sug_points

    def suggest(self, num_points: int | None = None) -> list[dict]:
        sug_points = []
        if len(self.outcome) == 0:
            print("\n\n\n\n\nRESTARTED OPTIMIZATION\n\n\n\n\n")
            sug_points = self.first_points(num_points)
        else:
            for tested_point in self.outcome:
                direction = 1
                for _ in range(2):
                    self.idx += 1
                    pitch_step = tested_point["pitch"] + (self.step * direction)
                    if not self.already_tested(pitch_step):
                        self.verified_points.append(pitch_step)
                        sug_points.append({
                            "_id": self.idx,
                            "m2_pitch": pitch_step
                        })
                        self.map_old2new_outcome[self.idx] = tested_point["center_x"]
                    direction *= -1
        return sug_points

    def ingest(self, points: list[dict]) -> None:
        self.outcome = []
        for tested_point in points:
            idx = tested_point["_id"]
            if idx > self.num_points:
                old_center = abs(self.map_old2new_outcome[idx])
                new_center = abs(tested_point["center_x"])
                if old_center > new_center:
                    self.outcome.append(tested_point)
            else:
                self.outcome.append(tested_point)

## Test scans

In [ ]:
optimization_problem = OptimizationProblem(
    readables=[m2],
    movables=[m2.pitch],
    optimizer=SearchOptimizer(),
    evaluation_function=FuzzyEvaluation(tiled_client)
)
RE(optimize(optimization_problem, iterations=5, n_points=1))

In [ ]:
RE(scan([m2], m2.pitch, -0.02, 0.02, num=10))

## Data Analysis with Tiled

In [ ]:
# Reprising the images of past scans
run = tiled_client.values()[-1]
reprise_simulation = run["primary/internal/m2_image"].read()
for image in reprise_simulation:
    beam2D = np.reshape(image, (101, 101))
    plot_beam(
        beam2D=beam2D,
        show_plot=True,
        fitType=3,
        textA=2,textB=5,
        x_range=1,x_range_min=-0.4,x_range_max=0.3,
        y_range=1,y_range_min=-0.4,y_range_max=0.3,
        zero_pad_x=2
    )

# for key in tiled_client:
#     run = tiled_client[key]
#     reprise_simulation = run["primary/internal/m2_image"].read()
#     for image in reprise_simulation:
#         beam2D = np.reshape(image, (101, 101))
#         plot_beam(
#             beam2D=beam2D,
#             show_plot=True,
#             fitType=3,
#             textA=2,textB=5,
#             x_range=1,x_range_min=-0.4,x_range_max=0.3,
#             y_range=1,y_range_min=-0.4,y_range_max=0.3,
#             zero_pad_x=2
#         )

In [ ]:
from prettytable import PrettyTable

run = tiled_client.values()[-1]
center_x = run["primary/internal/m2_center_x"].read()
center_y = run["primary/internal/m2_center_y"].read()
width = run["primary/internal/m2_width"].read()
height = run["primary/internal/m2_height"].read()
t = PrettyTable(['Seq Num', 'Center X', 'Center Y', "Width", "Height"])
for idx, (x, y, w, h) in enumerate(zip(center_x, center_y, width, height)):
    t.add_row([idx, x, y, w, h])
print(t)